In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# Load daily wide revenue data
df = pd.read_csv("daily_wide_revenue.csv")

df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
print(df.head())
print("Columns:", df.columns.tolist())

Dataset shape: (180, 4)
        date     North     South      West
0 2025-01-01  50651.59  48937.65  60109.02
1 2025-01-02  54431.06  52101.37  62082.51
2 2025-01-03  62516.51  53274.36  57028.24
3 2025-01-04  67241.10  60980.94  65800.24
4 2025-01-05  66825.20  66908.57  69358.06
Columns: ['date', 'North', 'South', 'West']


In [2]:
intervention_date = pd.Timestamp("2025-05-01")

pre = df[df["date"] < intervention_date].copy()
post = df[df["date"] >= intervention_date].copy()

print("Pre-treatment rows:", len(pre))
print("Post-treatment rows:", len(post))
print("Pre-treatment dates:", pre["date"].min(), "to", pre["date"].max())
print("Post-treatment dates:", post["date"].min(), "to", post["date"].max())

Pre-treatment rows: 120
Post-treatment rows: 60
Pre-treatment dates: 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Post-treatment dates: 2025-05-01 00:00:00 to 2025-06-29 00:00:00


In [3]:
from scipy.optimize import minimize
import numpy as np

# Pre-treatment data
X_pre = pre[["South", "West"]].to_numpy()
y_pre = pre["North"].to_numpy()

# Minimize pre-treatment mean squared error
def objective(weights):
    prediction = X_pre @ weights
    return np.mean((y_pre - prediction) ** 2)

result = minimize(
    objective,
    x0=np.array([0.5, 0.5]),
    method="SLSQP",
    bounds=[(0, 1), (0, 1)],
    constraints=[
        {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    ],
    options={"maxiter": 1000, "ftol": 1e-12}
)

if not result.success:
    raise RuntimeError(result.message)

south_weight, west_weight = result.x
pre_mse = result.fun

print(f"South weight: {south_weight:.4f}")
print(f"West weight: {west_weight:.4f}")
print(f"Total weight: {result.x.sum():.4f}")
print(f"Pre-treatment MSE: {pre_mse:.2f}")

South weight: 0.5000
West weight: 0.5000
Total weight: 1.0000
Pre-treatment MSE: 36627510.23


In [4]:
print("Optimization success:", result.success)
print("Optimizer message:", result.message)

# Check MSE at different donor weights
for w_south in [0.0, 0.25, 0.5, 0.75, 1.0]:
    weights = np.array([w_south, 1 - w_south])
    mse = np.mean((y_pre - X_pre @ weights) ** 2)
    print(
        f"South={weights[0]:.2f}, "
        f"West={weights[1]:.2f}, "
        f"MSE={mse:.2f}"
    )

Optimization success: True
Optimizer message: Optimization terminated successfully
South=0.00, West=1.00, MSE=80092516.26
South=0.25, West=0.75, MSE=48371671.90
South=0.50, West=0.50, MSE=36627510.23
South=0.75, West=0.25, MSE=44860031.26
South=1.00, West=0.00, MSE=73069234.99


In [5]:
df["synthetic_north"] = (
    south_weight * df["South"]
    + west_weight * df["West"]
)

df["gap"] = df["North"] - df["synthetic_north"]

post_results = df[df["date"] >= intervention_date]

average_effect = post_results["gap"].mean()
cumulative_effect = post_results["gap"].sum()

print(f"Post-treatment average effect: {average_effect:.2f}")
print(f"Post-treatment cumulative gap: {cumulative_effect:.2f}")

Post-treatment average effect: 2537.58
Post-treatment cumulative gap: 152254.85


In [6]:
baseline_results = pd.DataFrame([{
    "scenario": "baseline",
    "intervention_date": "2025-05-01",
    "south_weight": south_weight,
    "west_weight": west_weight,
    "pre_treatment_mse": pre_mse,
    "post_treatment_average_effect": average_effect,
    "post_treatment_cumulative_gap": cumulative_effect
}])

baseline_results.to_csv(
    "scm_baseline_results.csv",
    index=False
)

print(baseline_results.to_string(index=False))
print("\nBaseline results CSV saved successfully.")

scenario intervention_date  south_weight  west_weight  pre_treatment_mse  post_treatment_average_effect  post_treatment_cumulative_gap
baseline        2025-05-01           0.5          0.5       3.662751e+07                    2537.580833                      152254.85

Baseline results CSV saved successfully.


In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("daily_wide_revenue.csv")
df["date"] = pd.to_datetime(df["date"])

# Split pre-treatment data
intervention_date = pd.Timestamp("2025-05-01")

pre = df[df["date"] < intervention_date].copy()
post = df[df["date"] >= intervention_date].copy()

print("Dataset shape:", df.shape)
print("Pre-treatment rows:", len(pre))
print("Post-treatment rows:", len(post))

Dataset shape: (180, 4)
Pre-treatment rows: 120
Post-treatment rows: 60


In [3]:

weight_combinations = [
    (0.0, 1.0),
    (0.1, 0.9),
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1),
    (1.0, 0.0)
]

results = []

for south_w, west_w in weight_combinations:
    synthetic_pre = (
        south_w * pre["South"].to_numpy()
        + west_w * pre["West"].to_numpy()
    )

    mse = np.mean(
        (pre["North"].to_numpy() - synthetic_pre) ** 2
    )

    results.append({
        "south_weight": south_w,
        "west_weight": west_w,
        "pre_treatment_mse": mse
    })

weight_results = pd.DataFrame(results)

print(weight_results.round(2).to_string(index=False))


 south_weight  west_weight  pre_treatment_mse
          0.0          1.0        80092516.26
          0.1          0.9        65006976.59
          0.2          0.8        53117706.15
          0.3          0.7        44424704.95
          0.4          0.6        38927972.97
          0.5          0.5        36627510.23
          0.6          0.4        37523316.72
          0.7          0.3        41615392.44
          0.8          0.2        48903737.39
          0.9          0.1        59388351.58
          1.0          0.0        73069234.99


In [4]:

# Day 3: Alternative donor combinations

donor_scenarios = {
    "South_only": {"South": 1.0, "West": 0.0},
    "West_only": {"South": 0.0, "West": 1.0},
    "South_West_equal": {"South": 0.5, "West": 0.5}
}

scenario_results = []

for scenario, weights in donor_scenarios.items():

    # Pre-treatment synthetic North
    synthetic_pre = (
        weights["South"] * pre["South"].to_numpy()
        + weights["West"] * pre["West"].to_numpy()
    )

    pre_mse = np.mean(
        (pre["North"].to_numpy() - synthetic_pre) ** 2
    )

    # Post-treatment synthetic North
    synthetic_post = (
        weights["South"] * post["South"].to_numpy()
        + weights["West"] * post["West"].to_numpy()
    )

    post_gap = post["North"].to_numpy() - synthetic_post

    scenario_results.append({
        "scenario": scenario,
        "south_weight": weights["South"],
        "west_weight": weights["West"],
        "pre_treatment_mse": pre_mse,
        "post_treatment_average_gap": np.mean(post_gap),
        "post_treatment_cumulative_gap": np.sum(post_gap)
    })

scenario_results_df = pd.DataFrame(scenario_results)

print(scenario_results_df.round(2).to_string(index=False))


        scenario  south_weight  west_weight  pre_treatment_mse  post_treatment_average_gap  post_treatment_cumulative_gap
      South_only           1.0          0.0        73069234.99                     8457.91                      507474.74
       West_only           0.0          1.0        80092516.26                    -3382.75                     -202965.04
South_West_equal           0.5          0.5        36627510.23                     2537.58                      152254.85


In [5]:

scenario_results_df.to_csv(
    "scm_alternative_donor_results.csv",
    index=False
)

print("Saved: scm_alternative_donor_results.csv")
print("Scenarios saved:", len(scenario_results_df))


Saved: scm_alternative_donor_results.csv
Scenarios saved: 3
